In [ ]:
# Cell 1 — clone/pull repo + install probe deps
!git clone https://github.com/Jaswanth-K1210/SDAM.git 2>/dev/null || (cd SDAM && git pull --no-edit origin main)
%cd SDAM
!pip install -q -r probe/requirements.txt
print("Setup complete")

In [ ]:
# Cell 2 — GPU check (expect A100)
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

In [ ]:
# Cell 3 — Download CLEVR v1.0 (images + scene graphs), ~18GB, once.
import os
if not os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"):
    os.makedirs("data", exist_ok=True)
    # Primary source (may be blocked in some regions). Fallbacks: Kaggle CLEVR
    # mirror, or HuggingFace datasets 'clevr'.
    !wget -q --show-progress https://dl.fbaipublicfiles.com/clevr/CLEVR_v1.0.zip -O data/CLEVR_v1.0.zip
    !cd data && unzip -q CLEVR_v1.0.zip && rm -f CLEVR_v1.0.zip
print("scenes present:", os.path.exists("data/CLEVR_v1.0/scenes/CLEVR_train_scenes.json"))
print("images present:", os.path.isdir("data/CLEVR_v1.0/images/train"))

In [ ]:
# Cell 4 — math layer test suites (must be GREEN before run_probe)
!python -m pytest tests/test_variance.py tests/test_decodability.py -v -s

In [ ]:
# Cell 5 — factor/cosine/gate/verdict test suites
!python -m pytest tests/test_clevr_factors.py tests/test_cosine_check.py tests/test_variance_gate.py tests/test_run_probe.py -v

In [ ]:
# Cell 6 — run the probe (prints cosine -> decodability -> variance -> verdict)
!python -m probe.run_probe

In [ ]:
# Cell 7 — display plots
from IPython.display import Image, display
for p in ["cosine_histogram.png", "concentration_bars.png"]:
    print(p); display(Image(p))

In [ ]:
# Cell 8 — full results JSON
print(open("probe_results.json").read())

In [ ]:
# Cell 9 — commit results to GitHub
from google.colab import userdata
import os
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
!git config --global user.email "jaswanthkoppisetty@gmail.com"
!git config --global user.name "Jaswanth-K1210"
!mkdir -p results_archive/probe
!cp -f probe_results.json cosine_histogram.png concentration_bars.png results_archive/probe/ 2>/dev/null || true
!git add results_archive/probe/
!git commit -m "results(probe): CLEVR DINOv2 feasibility probe run + verdict" || echo "nothing to commit"
if token:
    os.system(f"git remote set-url origin https://{token}@github.com/Jaswanth-K1210/SDAM.git")
    os.system("git push origin main")
    print("Pushed.")
else:
    print("Add GITHUB_TOKEN to Colab Secrets then re-run.")